In [681]:
# Initialize Otter
import otter
grader = otter.Notebook("week4.ipynb")

# HW4: Constraint Programming
#### CS-31: Foundations for Artificial Intelligence
*Point breakdown: 40 autograded · 60 manually graded · 10 collaboration statement = 100*


*You are allowed to work in groups for the lab. You must write anything that will be turned in -- all code and all written solutions -- on your own. You must list all people and resources that you interacted with in the Collaboration Statement below. See [course policies](https://www.cs.tufts.edu/comp/31/policy.html) for more information.*

--


In this lab, you'll be getting hands-on practice with an industry standard combinatorial optimization library - [Google OR-Tools](https://developers.google.com/optimization/). Essentially libraries like these are behind the scenes of any scheduling or optimization program you may encounter. They provide the raw building blocks for solving optimization problems, that domain experts (such as yourselves) use to turn into more specialized tools relevant to your domain. It's a tool to *build* the tools! OR-Tools has many optimization algorithms built in that very smart people have spent decades optimizing to be as fast as possible - and you're going to learn how to use them to solve Constraint Satisfaction Problems. This is wrapped up in the [CP-SAT](https://developers.google.com/optimization/cp/cp_solver) portion of their library.

Finally, I'll ask you to read an article on the impacts of an automated scheduling tool on how hotel housekeepers do their job. You may think that "automated scheduling" isn't as fancy or eye-popping as the newest chatbots and Large Language Models - but it is *still* a form of automated decision-making where we have to think critically about performance measures and outcomes. It's all connected, no matter how "simple" the tools are.

**AI / Source Disclosure**
- I used Claude to generate the `NursesCalendarPrinter`. It only controls what the output looks like so you can understand it better, not any of the underlying algorithms or CSP core.
- Some of the prose and code for the nurse scheduling section are adapted from OR-Tools official tutorial.
- All other prose and code was written by me.


### **Table of Contents**

1) Coding: CP-SAT Basics
2) Coding: CP-SAT for Scheduling
3) Reading: HotSOS Struggles
4) Collaboration Statement


## **1. CP-SAT Basics**


Here's a very brief introduction of how you use OR-Tools for constraint satisfaction problems. I would highly recommend reading through their [examples and documentation](https://developers.google.com/optimization/cp/cp_example).

In [682]:
# OR tools has tools for many types of optimization problems
# Right now, we care about Constraint Programming, so we import CP Model
from ortools.sat.python import cp_model

In [683]:
# We create a new CP model
model = cp_model.CpModel()

In [684]:
# Oftentimes, we use dictionaries to keep track of our variables
variables = {}

# We create a new boolean variable with the name "A"
# And store it in our dictionary with a key of 0
variables[0] = model.new_bool_var("A")

# Do the same for some other variables
variables[1] = model.new_bool_var("B")
variables[2] = model.new_bool_var("C")
variables[3] = model.new_bool_var("D")

In [685]:
# Let's print our variables
variables

# A(0..1) just means that "A is a boolean variable"

{0: A(0..1), 1: B(0..1), 2: C(0..1), 3: D(0..1)}

Now let's `add` a constraint. Let's make it so that `B` (which is in `variables[1]`) must be equal to 1.

In [686]:
# B should always be 1
model.add(variables[1] == 1)

Now, we want to have constraints across *multiple* variables (e.g. exactly one of these variables must be one, etc)

In general, the usual way to do this is to create subsets of variables (e.g. by building a list), and then putting a constraint on that subset. Here's an example - let's say I want to constrain all of the even variables (e.g. "A" and "C") such that exactly one of those can be true (not both).

In [687]:
# Build our list of what we want to constrain...
to_constrain = []

# Iterate through our variables
for i in range(len(variables)):

    # Get our even variables
    if i % 2 == 0:
        to_constrain.append(variables[i])

# These are the variables we want to constrain
print(to_constrain)

[A(0..1), C(0..1)]


In [688]:
# Then: we put a constraint on those variables!
model.add_exactly_one(to_constrain)

This means that in our final solution - either "A" or "C" has to be one (but not both).

Alright, so now we have the following constraints:
- "A" or "C" has to be one (but not both).
- "B" must be one.

Let's solve our constraint problem!

In [689]:
# We create a new solver...
solver = cp_model.CpSolver()

# and solve!
status = solver.solve(model)

# Print our result:
print(status)

CpSolverStatus.OPTIMAL


Congrats, you solved your first CSP! The solver, when you print it, returns a status code, which can be one of:

- `OPTIMAL` - An optimal feasible solution was found.
- `FEASIBLE` - A feasible solution was found, but we don't know if it's optimal.
- `INFEASIBLE` - the problem was proven to be infeasible (i.e. impossible to solve.)

For constraint satisfaction where we're only concerned about finding *a* solution, we can treat `OPTIMAL` and `FEASIBLE` as the same thing.

We can then list off the solution that our solver found by using `solver.value()`

In [690]:
# Iterate through all of our variables
for i in range(len(variables)):
    print(variables[i], "=", solver.value(variables[i]))

A = 1
B = 1
C = 0
D = 0


We see that all of our constraints are satisfied with this solution.

Now I want you to consider the following CSP setup...

In [691]:
# Create a fresh model and variables
model = cp_model.CpModel()

variables = {}

variables[0] = model.new_bool_var("A")
variables[1] = model.new_bool_var("B")
variables[2] = model.new_bool_var("C")
variables[3] = model.new_bool_var("D")

# Same as before...
to_constrain = []
for i in range(len(variables)):

    # Get our even variables
    if i % 2 == 0:
        to_constrain.append(variables[i])

# Now let's add these constraints
model.add_at_least_one(to_constrain)
model.add(variables[0] == 0)
model.add(variables[2] == 0)

Before running the next cell, what do you think the CSP solver will return, and why?

In [692]:
solver = cp_model.CpSolver()
status = solver.solve(model)
print(status)

CpSolverStatus.INFEASIBLE


#### **Exercise 1.1 (Written): Solver Result (10 pts)**

Why did our solver give the above result? Explain in plain English in two or three sentences.

<!-- BEGIN QUESTION -->

This is infeasible because the constraints that we have set are contradictory. The first constrain applies to all variables with even number indexes such as A and C, and it constraints them to have at least one of them be 1. The second and third constraints make variables[0] and variables[2] which are A and C to be 0, which is contradictory to the first constraint since it is impossible for the even number variables to be 0 and have at least one equal to 1.

<!-- END QUESTION -->

### **1.1 Dictionaries indexed on Tuples**


Finally, one more bit of Python trivia that's good to know is that you can use *tuples* as keys for dictionaries.

In [693]:
mult_table = {}

for i in range(4):
    for j in range(4):
        mult_table[(i,j)] = i*j

print(mult_table)
print(mult_table[(3,3)])
print(mult_table[(2,1)])
print(mult_table[(1,2)])

{(0, 0): 0, (0, 1): 0, (0, 2): 0, (0, 3): 0, (1, 0): 0, (1, 1): 1, (1, 2): 2, (1, 3): 3, (2, 0): 0, (2, 1): 2, (2, 2): 4, (2, 3): 6, (3, 0): 0, (3, 1): 3, (3, 2): 6, (3, 3): 9}
9
2
2


This is good to know because this is often how we keep track of boolean variables in CP-SAT. For example, consider our seating problem we talked about in lecture.

In [694]:
model = cp_model.CpModel()

seating_assignments = {}


for person in ["Alice","Bob","Carly","Dan"]:
    for seat in [0,1,2,3]:
        for food in ["Steak", "Falafel"]:
            seating_assignments[ (person, seat, food) ] = model.new_bool_var(f"{person}_{seat}_{food}")

# Print out our dictionary
print(seating_assignments)

# Notice how we have 4*4*2 = 32 boolean vars
print(len(seating_assignments))

# Print each of the variables
# Boolean representing: Alice is at seat 0 and served Steak
print(seating_assignments[('Alice', 0, "Steak")])       

# Boolean representing: Alice is at seat 1 and served Falafel
print(seating_assignments[('Alice', 1, "Falafel")])       

# Boolean representing: Bob is at seat 3 and served Steak
print(seating_assignments[('Bob', 3, "Steak")])    
# etc.     

{('Alice', 0, 'Steak'): Alice_0_Steak(0..1), ('Alice', 0, 'Falafel'): Alice_0_Falafel(0..1), ('Alice', 1, 'Steak'): Alice_1_Steak(0..1), ('Alice', 1, 'Falafel'): Alice_1_Falafel(0..1), ('Alice', 2, 'Steak'): Alice_2_Steak(0..1), ('Alice', 2, 'Falafel'): Alice_2_Falafel(0..1), ('Alice', 3, 'Steak'): Alice_3_Steak(0..1), ('Alice', 3, 'Falafel'): Alice_3_Falafel(0..1), ('Bob', 0, 'Steak'): Bob_0_Steak(0..1), ('Bob', 0, 'Falafel'): Bob_0_Falafel(0..1), ('Bob', 1, 'Steak'): Bob_1_Steak(0..1), ('Bob', 1, 'Falafel'): Bob_1_Falafel(0..1), ('Bob', 2, 'Steak'): Bob_2_Steak(0..1), ('Bob', 2, 'Falafel'): Bob_2_Falafel(0..1), ('Bob', 3, 'Steak'): Bob_3_Steak(0..1), ('Bob', 3, 'Falafel'): Bob_3_Falafel(0..1), ('Carly', 0, 'Steak'): Carly_0_Steak(0..1), ('Carly', 0, 'Falafel'): Carly_0_Falafel(0..1), ('Carly', 1, 'Steak'): Carly_1_Steak(0..1), ('Carly', 1, 'Falafel'): Carly_1_Falafel(0..1), ('Carly', 2, 'Steak'): Carly_2_Steak(0..1), ('Carly', 2, 'Falafel'): Carly_2_Falafel(0..1), ('Carly', 3, 'Steak

## **2. CP-SAT for Scheduling**

**Note:** *This is a modified version of the [Employee Scheduling](https://developers.google.com/optimization/scheduling/employee_scheduling) tutorial in the official OR-Tools documentation. I've made my own slight edits to their code and prose to better demonstrate the boolean variables we're subsetting over, as well as to make it more interactive.*

Let's consider a scheduling problem.

A hospital supervisor needs to create a schedule for six nurses over a four-day period, subject to the following conditions:

- Each day is divided into three 8-hour shifts.
- Every day, each shift is assigned to a single nurse, and no nurse works more than one shift.
- Each nurse is assigned to exactly two shifts during the four-day period.

Of course, in reality, the supervisor would most likely use an existing application with a nice graphical user interface. But, *how do you think the application works when you look at the code*? It looks a lot like this! And yes, in real life hospitals are bigger with more nurses, shifts, and days - but as with many things in programming, once you figure out how to solve it for a small *n*, you can solve it for a big *n*.

The following sections present a solution to the nurse scheduling problem.

In [695]:
from ortools.sat.python import cp_model

Create our data for the example.

In [696]:
num_nurses = 6 # You'll set this to 2 later
num_shifts = 3
num_days = 4
all_nurses = range(num_nurses)
all_shifts = range(num_shifts)
all_days = range(num_days)

In [697]:
model = cp_model.CpModel()

The following code creates a dictionary of variables.

The dictionary defines assignments for shifts to nurses as follows: `shifts[(n, d, s)]` equals 1 if shift s is assigned to nurse n on day d, and 0 otherwise.

In [698]:
shifts = {}

for n in all_nurses:
    for d in all_days:
        for s in all_shifts:
            shifts[(n, d, s)] = model.new_bool_var(f"shift_n{n}_d{d}_s{s}")


print(shifts)

# Since there are 6 nurses, 3 shifts per day, and 4 days
# There are 6*3*4 boolean variables
print(len(shifts))

{(0, 0, 0): shift_n0_d0_s0(0..1), (0, 0, 1): shift_n0_d0_s1(0..1), (0, 0, 2): shift_n0_d0_s2(0..1), (0, 1, 0): shift_n0_d1_s0(0..1), (0, 1, 1): shift_n0_d1_s1(0..1), (0, 1, 2): shift_n0_d1_s2(0..1), (0, 2, 0): shift_n0_d2_s0(0..1), (0, 2, 1): shift_n0_d2_s1(0..1), (0, 2, 2): shift_n0_d2_s2(0..1), (0, 3, 0): shift_n0_d3_s0(0..1), (0, 3, 1): shift_n0_d3_s1(0..1), (0, 3, 2): shift_n0_d3_s2(0..1), (1, 0, 0): shift_n1_d0_s0(0..1), (1, 0, 1): shift_n1_d0_s1(0..1), (1, 0, 2): shift_n1_d0_s2(0..1), (1, 1, 0): shift_n1_d1_s0(0..1), (1, 1, 1): shift_n1_d1_s1(0..1), (1, 1, 2): shift_n1_d1_s2(0..1), (1, 2, 0): shift_n1_d2_s0(0..1), (1, 2, 1): shift_n1_d2_s1(0..1), (1, 2, 2): shift_n1_d2_s2(0..1), (1, 3, 0): shift_n1_d3_s0(0..1), (1, 3, 1): shift_n1_d3_s1(0..1), (1, 3, 2): shift_n1_d3_s2(0..1), (2, 0, 0): shift_n2_d0_s0(0..1), (2, 0, 1): shift_n2_d0_s1(0..1), (2, 0, 2): shift_n2_d0_s2(0..1), (2, 1, 0): shift_n2_d1_s0(0..1), (2, 1, 1): shift_n2_d1_s1(0..1), (2, 1, 2): shift_n2_d1_s2(0..1), (2, 2, 0)

### **2.1 Assigning nurses to shifts**

Next, we show how to assign nurses to shifts subject to the following constraints:

- Each shift is assigned to a single nurse per day.
- Each nurse works at most one shift per day.

Here's the code that creates the first condition.

In [699]:
# Each shift is assigned to a single nurse per day
for d in all_days:
    for s in all_shifts:

        nurses_on_shift = []

        for n in all_nurses:
            nurses_on_shift.append(shifts[(n, d, s)])

        # Print out the sets we're putting constraints on
        print(nurses_on_shift)
        model.add_exactly_one(nurses_on_shift)

[shift_n0_d0_s0(0..1), shift_n1_d0_s0(0..1), shift_n2_d0_s0(0..1), shift_n3_d0_s0(0..1), shift_n4_d0_s0(0..1), shift_n5_d0_s0(0..1)]
[shift_n0_d0_s1(0..1), shift_n1_d0_s1(0..1), shift_n2_d0_s1(0..1), shift_n3_d0_s1(0..1), shift_n4_d0_s1(0..1), shift_n5_d0_s1(0..1)]
[shift_n0_d0_s2(0..1), shift_n1_d0_s2(0..1), shift_n2_d0_s2(0..1), shift_n3_d0_s2(0..1), shift_n4_d0_s2(0..1), shift_n5_d0_s2(0..1)]
[shift_n0_d1_s0(0..1), shift_n1_d1_s0(0..1), shift_n2_d1_s0(0..1), shift_n3_d1_s0(0..1), shift_n4_d1_s0(0..1), shift_n5_d1_s0(0..1)]
[shift_n0_d1_s1(0..1), shift_n1_d1_s1(0..1), shift_n2_d1_s1(0..1), shift_n3_d1_s1(0..1), shift_n4_d1_s1(0..1), shift_n5_d1_s1(0..1)]
[shift_n0_d1_s2(0..1), shift_n1_d1_s2(0..1), shift_n2_d1_s2(0..1), shift_n3_d1_s2(0..1), shift_n4_d1_s2(0..1), shift_n5_d1_s2(0..1)]
[shift_n0_d2_s0(0..1), shift_n1_d2_s0(0..1), shift_n2_d2_s0(0..1), shift_n3_d2_s0(0..1), shift_n4_d2_s0(0..1), shift_n5_d2_s0(0..1)]
[shift_n0_d2_s1(0..1), shift_n1_d2_s1(0..1), shift_n2_d2_s1(0..1), sh

Let's take a closer look at that first set that we put a constraint over.

`[shift_n0_d0_s0(0..1), shift_n1_d0_s0(0..1), shift_n2_d0_s0(0..1), shift_n3_d0_s0(0..1), shift_n4_d0_s0(0..1), shift_n5_d0_s0(0..1)]`

This set showcases all the possible nurses that could be assigned to Day 0, Shift 0. We put an `exactly one` constraint on this set, which means that in this set - one and only one of these variables can be set to true. This encodes the constraint that "Only a single nurse can work Day 0, Shift 0".

Notice how we then apply this constraint to every other `day x shift` combination.

#### **Exercise 2.1 (Written): Nurses on Shift** **(10 pts)**

For the set: 
`[shift_n0_d0_s0(0..1), shift_n1_d0_s0(0..1), shift_n2_d0_s0(0..1), shift_n3_d0_s0(0..1), shift_n4_d0_s0(0..1), shift_n5_d0_s0(0..1)]`


- In English, what would it mean if more than one of these boolean variables was true in the final solution?
- In English, what would it mean if none of these boolean variables were true in the final solution?

<!-- BEGIN QUESTION -->

- The set is all of the possibilities of nurses for day 0 shift 0 where there are 6 different nurses available for that shift. If more than one of the boolean variables was true, it would violate the constraint that we set which is that each day each shift can only be assigned to one nurse. Therefore, if this happens in the final solution it would be incorrect and infeasible.
  
- If none of the boolean variables were true, then no nurses would be assigned to the shift on day 0. This would also violate the constraint since each day each shift must have one nurse assigned to it. So the solution would be infeasible.

<!-- END QUESTION -->

Now we want to encode the constraint that each nurse works at most one shift per day.

In [700]:
# A nurse can work only one shift a day
for n in all_nurses:
    for d in all_days:

        shifts_worked_per_day = []

        for s in all_shifts:
            shifts_worked_per_day.append(shifts[(n, d, s)])

        print(shifts_worked_per_day)
        
        model.add_at_most_one(shifts_worked_per_day)

[shift_n0_d0_s0(0..1), shift_n0_d0_s1(0..1), shift_n0_d0_s2(0..1)]
[shift_n0_d1_s0(0..1), shift_n0_d1_s1(0..1), shift_n0_d1_s2(0..1)]
[shift_n0_d2_s0(0..1), shift_n0_d2_s1(0..1), shift_n0_d2_s2(0..1)]
[shift_n0_d3_s0(0..1), shift_n0_d3_s1(0..1), shift_n0_d3_s2(0..1)]
[shift_n1_d0_s0(0..1), shift_n1_d0_s1(0..1), shift_n1_d0_s2(0..1)]
[shift_n1_d1_s0(0..1), shift_n1_d1_s1(0..1), shift_n1_d1_s2(0..1)]
[shift_n1_d2_s0(0..1), shift_n1_d2_s1(0..1), shift_n1_d2_s2(0..1)]
[shift_n1_d3_s0(0..1), shift_n1_d3_s1(0..1), shift_n1_d3_s2(0..1)]
[shift_n2_d0_s0(0..1), shift_n2_d0_s1(0..1), shift_n2_d0_s2(0..1)]
[shift_n2_d1_s0(0..1), shift_n2_d1_s1(0..1), shift_n2_d1_s2(0..1)]
[shift_n2_d2_s0(0..1), shift_n2_d2_s1(0..1), shift_n2_d2_s2(0..1)]
[shift_n2_d3_s0(0..1), shift_n2_d3_s1(0..1), shift_n2_d3_s2(0..1)]
[shift_n3_d0_s0(0..1), shift_n3_d0_s1(0..1), shift_n3_d0_s2(0..1)]
[shift_n3_d1_s0(0..1), shift_n3_d1_s1(0..1), shift_n3_d1_s2(0..1)]
[shift_n3_d2_s0(0..1), shift_n3_d2_s1(0..1), shift_n3_d2_s2(0.

Again, let's take a closer look at what this block of code is saying. Let's look at the first set we put a constraint over.

`[shift_n0_d0_s0(0..1), shift_n0_d0_s1(0..1), shift_n0_d0_s2(0..1)]`

This set shows the variables for Nurse 0 on Day 0, indicating whether they work during shifts 0, 1, or 2.  We put an `at most one` constraint on this set, which means that in this set - at most one of these variables can be set to true. This encodes the constraint that "Nurse 0 can work only one shift on Day 0".

Notice how we then apply this constraint to every other `nurse x day` combination.

#### **Exercise 2.2 (Written): Shifts Worked Per Day** **(10 pts)**

For the set: 
`[shift_n0_d0_s0(0..1), shift_n0_d0_s1(0..1), shift_n0_d0_s2(0..1)]`

- In English, what would it mean if more than one of these boolean variables was true in the final solution?
- In English, what would it mean if none of these boolean variables were true in the final solution?
- Why *don't* we use the `exactly_one` constraint here?

<!-- BEGIN QUESTION -->

- This is the set of all possibilities for nurse 0 on day 0, so the nurse would have 3 possible shifts on day 0. If more than one of these boolean variables were true, it violates our constraints where each day no nurse works more than one shift. So the final solution would be infeasible.

- If none of the boolean variables were true, it means that nurse 0 does not work on day 0 which is also a possibility. As long as a nurse is assigned on these shifts, it would not violate the constraint and it does not have to be nurse 0 in this case. 
  
- We don't use `exactly_one` because everything being false is also a possible outcome here but `exactly_one` eliminats this outcome.

<!-- END QUESTION -->

### **2.2 Distributing Shifts Equally**

Finally, we want to make sure that each nurse gets an equal number of shifts across the days.

Here's how we do that in code.

First, we need to figure out how many shifts each nurse should work per week.

In [701]:
# The total of shifts that need to be scheduled
total_shifts_need_filled = num_shifts * num_days

shifts_per_nurse = total_shifts_need_filled // num_nurses
# // means integer division, because a nurse must have a whole number amount of shifts

print(shifts_per_nurse)

2


We see that to keep things fair, each nurse should only work 2 shifts across all of the days.

Let's encode this.

In [702]:
for n in all_nurses:
    shifts_worked = []

    for d in all_days:
        for s in all_shifts:
            shifts_worked.append(shifts[(n, d, s)])

    print(shifts_worked)

    # Each nurse should only work the required number of shifts per nurse
    # You'll remove this and see what happens later in the exercise:
    # model.add(sum(shifts_worked) == shifts_per_nurse)

[shift_n0_d0_s0(0..1), shift_n0_d0_s1(0..1), shift_n0_d0_s2(0..1), shift_n0_d1_s0(0..1), shift_n0_d1_s1(0..1), shift_n0_d1_s2(0..1), shift_n0_d2_s0(0..1), shift_n0_d2_s1(0..1), shift_n0_d2_s2(0..1), shift_n0_d3_s0(0..1), shift_n0_d3_s1(0..1), shift_n0_d3_s2(0..1)]
[shift_n1_d0_s0(0..1), shift_n1_d0_s1(0..1), shift_n1_d0_s2(0..1), shift_n1_d1_s0(0..1), shift_n1_d1_s1(0..1), shift_n1_d1_s2(0..1), shift_n1_d2_s0(0..1), shift_n1_d2_s1(0..1), shift_n1_d2_s2(0..1), shift_n1_d3_s0(0..1), shift_n1_d3_s1(0..1), shift_n1_d3_s2(0..1)]
[shift_n2_d0_s0(0..1), shift_n2_d0_s1(0..1), shift_n2_d0_s2(0..1), shift_n2_d1_s0(0..1), shift_n2_d1_s1(0..1), shift_n2_d1_s2(0..1), shift_n2_d2_s0(0..1), shift_n2_d2_s1(0..1), shift_n2_d2_s2(0..1), shift_n2_d3_s0(0..1), shift_n2_d3_s1(0..1), shift_n2_d3_s2(0..1)]
[shift_n3_d0_s0(0..1), shift_n3_d0_s1(0..1), shift_n3_d0_s2(0..1), shift_n3_d1_s0(0..1), shift_n3_d1_s1(0..1), shift_n3_d1_s2(0..1), shift_n3_d2_s0(0..1), shift_n3_d2_s1(0..1), shift_n3_d2_s2(0..1), shift_

One last time, let's take a look at that first set we're putting a constraint on:

`[shift_n0_d0_s0(0..1), shift_n0_d0_s1(0..1), shift_n0_d0_s2(0..1), shift_n0_d1_s0(0..1), ... shift_n0_d3_s0(0..1), shift_n0_d3_s1(0..1), shift_n0_d3_s2(0..1)]`

This set shows all the possible shifts that Nurse 0 can work across *all* days. What we do is we take the `sum` of all these variables. Since each variable is 1 if and only if Nurse 0 works Shift `s` on Day `d`, and zero otherwise, the `sum` of these variables represents the total number of shifts that Nurse 0 works. We set this sum to equal `shifts_per_nurse` (e.g. 2), as each nurse should only work 2 shifts.

Notice how we then apply this constraint to all the nurses for every possible shift they can work.

#### **Exercise 2.3 (Written): Shifts Worked** **(10 pts)**

For the set: 
`[shift_n0_d0_s0(0..1), shift_n0_d0_s1(0..1), shift_n0_d0_s2(0..1), shift_n0_d1_s0(0..1), ... shift_n0_d3_s0(0..1), shift_n0_d3_s1(0..1), shift_n0_d3_s2(0..1)]`

- In English, what would it mean if 5 of these boolean variables were true in the final solution?
- In English, what would it mean if none of these boolean variables were true in the final solution?

<!-- BEGIN QUESTION -->

- This set is all the possible shifts that nurse 0 can take in the 4 day period. If 5 of them are true, nurse 0 would work for 5 different shifts during the 4 day period which violates the constraint of working 2 shifts max for the 4 day period. Therefore, it would mean that the final solution is infeasible.

- If none of the boolean variables were true, nurse 0 did not work any shift in the 4 day period. This could not work because there are 6 nurses and 12 shifts in total. Thus, if one nurse does not work in the 4 day period, other nurse has to work more than 2 shifts which does not satisfy the constraint. So the final solution would also be infeasible and incorrect if none of the variables were true.

<!-- END QUESTION -->

### **2.3 Solving the Problem**

Now that all of our constraints are set: we can actually solve our scheduling problem!

There are many, many possible solutions to the problem - and we only care about finding a few. Enumerating all of the solutions would take a large amount of time, so we define a special solution printer that allows us to control how many solutions we can see. In addition, this printer will format the solutions in a nice calendar format. **You can ignore the following code cell, it's just setting up the solution printer. Feel free to hide this cell.**

In [703]:
# NOTE: You can safely ignore the code in this cell. This is just so we can limit the number of 
# solutions we surface, as well as print them out in a nice calendar format.


class NursesCalendarPrinter(cp_model.CpSolverSolutionCallback):
    """Print intermediate solutions as a grid: rows = shifts, columns = days,
    cells = nurse number(s) assigned to that shift on that day."""
 
    def __init__(self, shifts, num_nurses, num_days, num_shifts, limit,
                 shift_names=None, seniority=None):
        cp_model.CpSolverSolutionCallback.__init__(self)
        self._shifts = shifts
        self._num_nurses = num_nurses
        self._num_days = num_days
        self._num_shifts = num_shifts
        self._solution_count = 0
        self._solution_limit = limit
 
        if shift_names is None:
            shift_names = ["Morning", "Afternoon", "Evening"]
        # Fall back to generic labels if names don't match the shift count.
        if len(shift_names) != num_shifts:
            shift_names = [f"Shift {s}" for s in range(num_shifts)]
        self._shift_names = list(shift_names)
 
        # seniority[n]: 1 = senior, 0 = junior. Juniors are marked with "*".
        if seniority is not None and len(seniority) != num_nurses:
            raise ValueError(
                f"seniority has {len(seniority)} entries; expected {num_nurses}"
            )
        self._seniority = seniority
 
    def _nurse_label(self, n):
        if self._seniority is not None and not self._seniority[n]:
            return f"{n}*"
        return str(n)
 
    def on_solution_callback(self):
        # stop_search() is asynchronous: with multiple workers, solutions
        # already in flight can still arrive after it's called. Ignore them.
        if self._solution_count >= self._solution_limit:
            return
        self._solution_count += 1
 
        # grid[s][d] -> cell text. Joins with "," if >1 nurse; "-" if none.
        grid = []
        for s in range(self._num_shifts):
            row = []
            for d in range(self._num_days):
                assigned = [
                    self._nurse_label(n)
                    for n in range(self._num_nurses)
                    if self.value(self._shifts[(n, d, s)])
                ]
                row.append(",".join(assigned) if assigned else "-")
            grid.append(row)
 
        header = [f"Day {d}" for d in range(self._num_days)]
        label_w = max(len(name) for name in self._shift_names)
        col_w = max([len(h) for h in header] +
                    [len(cell) for row in grid for cell in row])
 
        def fmt_row(label, cells):
            return (label.ljust(label_w) + " | " +
                    " | ".join(c.center(col_w) for c in cells))
 
        header_line = fmt_row("", header)
        print(f"Solution {self._solution_count}")
        print(header_line)
        print("-" * len(header_line))
        for name, row in zip(self._shift_names, grid):
            print(fmt_row(name, row))
        if self._seniority is not None:
            print("(* = junior)")
        print()
 
        if self._solution_count >= self._solution_limit:
            print(f"Stop search after {self._solution_limit} solutions")
            self.stop_search()
 
    def solutionCount(self):
        return self._solution_count


#### **Completing the Solution**

Okay, now back to the important stuff you *should* know.

We've set up our model - so now we can set up our `solver` to actually give us some solutions.

In [704]:
# Set up our solver
solver = cp_model.CpSolver()

# Tell our solver we want to systematically enumerate solutions
solver.parameters.enumerate_all_solutions = True

# Only get the first 5 solutions
solution_limit = 5

# Print them out using our special printer
solution_printer = NursesCalendarPrinter(
    shifts, num_nurses, num_days, num_shifts, solution_limit,
)

# Solve
status = solver.solve(model, solution_printer)

print(status)

Solution 1
          | Day 0 | Day 1 | Day 2 | Day 3
-----------------------------------------
Morning   |   2   |   0   |   2   |   2  
Afternoon |   0   |   2   |   0   |   0  
Evening   |   1   |   1   |   1   |   1  

Solution 2
          | Day 0 | Day 1 | Day 2 | Day 3
-----------------------------------------
Morning   |   2   |   3   |   2   |   2  
Afternoon |   0   |   2   |   0   |   0  
Evening   |   1   |   1   |   1   |   1  

Solution 3
          | Day 0 | Day 1 | Day 2 | Day 3
-----------------------------------------
Morning   |   2   |   3   |   2   |   2  
Afternoon |   0   |   0   |   0   |   0  
Evening   |   1   |   1   |   1   |   1  

Solution 4
          | Day 0 | Day 1 | Day 2 | Day 3
-----------------------------------------
Morning   |   2   |   3   |   2   |   2  
Afternoon |   0   |   2   |   0   |   0  
Evening   |   1   |   0   |   1   |   1  

Solution 5
          | Day 0 | Day 1 | Day 2 | Day 3
-----------------------------------------
Morning   |   2  

Congrats, you just solved your first *real-ish* CSP! You can see that in the solutions that were surfaced, all of our constraints we set up were followed. The solver also returns `FEASIBLE`. For constraint problems, this just means we found a solution that satisfied all of our constraints.

#### **Exercise 2.4 (Written): Messing With Constraints** **(10 pts)**

Answer the following questions.

- When we first defined our problem a few cells ago, set the number of nurses to be 2. Predict what you think will happen when we run the solve. Then Restart and run all the cells up until this point. Write down what you notice.
- In the cell that sets up the constraints to distribute shifts equally, comment out `model.add(sum(shifts_worked) == shifts_per_nurse)` (e.g. remove the distribution constraint). First, predict what will happen when we run the solver. Then Restart and run the cells up to this point. Write down what you notice.

<!-- BEGIN QUESTION -->

- if we set the number of nurses to be 2, it would be impossible to satisfy all the constraints since there would not be enough nurses if each nurse only work for 2 shifts in the 4 day period. Therefore, the final result would be infeasible. Specifically, if there are two nurses, they can only work for 4 shifts which is not enough for the 12 shifts during the four days. As predicted, the model returned infeasible because it fails to satisfy all the constraints. One of the things I noticed was that the the set for all the possible shifts decreased which is to be expected since the number of nurses also decreased.

- `shifts_worked == shifts_per_nurse` is essentially the constraint where each nurse can only work for 2 shifts max in four days and commenting it out would remove the constraint. This meant that each nurse can work for more shifts and not all nurses will have to work in the four day period. I predict that not all nurses will work in the final solution. As expected, only nurse 0, 1, 2, 3 worked in the final solutions and each nurse worked more shifts as less number of nurses work in the four days period. 

<!-- END QUESTION -->

### **2.4 Accounting for Uneven Shift Distribution**


Let's set up our problem again, but be a little more realistic. I chose special values for the number of nurses, number of shifts, and number of days such that the number of nurses cleanly divides the total number of shifts that need to be filled. However, that will most likely not be the case for many scenarios. We will now account for that.

First, let's set up our solver again with a fresh model, and add back all of our constraints we had. 

**Make sure to always set up a fresh model, or else you might have constraints that hold over from the previous model! When in doubt, Restart+Run All!**

In [705]:
num_nurses = 4 # <- Now notice that we can't cleanly divide days*shifts into nurses!
num_shifts = 3
num_days = 3
all_nurses = range(num_nurses)
all_shifts = range(num_shifts)
all_days = range(num_days)

model = cp_model.CpModel()

shifts = {}

for n in all_nurses:
    for d in all_days:
        for s in all_shifts:
            shifts[(n, d, s)] = model.new_bool_var(f"shift_n{n}_d{d}_s{s}")

# A shift is assigned to exactly one nurse
for d in all_days:
    for s in all_shifts:

        shifts_worked = []
        for n in all_nurses:
            shifts_worked.append(shifts[(n, d, s)])

        model.add_exactly_one(shifts_worked)

# Nurse can work only one shift a day
for n in all_nurses:
    for d in all_days:

        shifts_worked = []

        for s in all_shifts:
            shifts_worked.append(shifts[(n, d, s)])
        
        model.add_at_most_one(shifts_worked)

Now, let's try to account for the shift distribution problem.

In [706]:
total_shifts_need_filled = num_shifts * num_days

# We can see that if we tried to divide the shifts needing to be filled among nurses
# equally, each nurse would need to work 2 shifts, plus some extra.
print(total_shifts_need_filled / num_nurses)

min_shifts_per_nurse = total_shifts_need_filled // num_nurses

# So we know that all nurses have to work AT LEAST two shifts
print(min_shifts_per_nurse)

2.25
2


In [707]:
# If the number of nurses divides the total number of shifts cleanly...
# Then the maximum and minimum is the same
if total_shifts_need_filled % num_nurses == 0:
    max_shifts_per_nurse = min_shifts_per_nurse

# If it doesn't, some nurses might need to work one extra shift (e.g. 3)
else:
    max_shifts_per_nurse = min_shifts_per_nurse + 1

print(max_shifts_per_nurse)

3


Now, we go back to work on our constraint(s) which took care of how many shifts each nurse could work in total.

In [708]:
for n in all_nurses:
    shifts_worked = []
    
    for d in all_days:
        for s in all_shifts:
            shifts_worked.append(shifts[(n, d, s)])

    # Remove the old constraint
    # REMOVED: ~model.add(sum(shifts_worked) == shifts_per_nurse)~

    # Add the constraints that min_shifts_per_nurse <= sum(shifts_worked) <= max_shifts_per_nurse
    # Nurse can work no less than min shifts, and no more than max shifts.
    model.add(min_shifts_per_nurse <= sum(shifts_worked))
    model.add(sum(shifts_worked) <= max_shifts_per_nurse)

And then we can solve our problem no issue! Notice how now, one unlucky nurse has to work one extra shift in each assignment.

In [709]:
solver = cp_model.CpSolver()
solver.parameters.enumerate_all_solutions = True
solution_limit = 5

solution_printer = NursesCalendarPrinter(
    shifts, num_nurses, num_days, num_shifts, solution_limit,
)
solver.solve(model, solution_printer)

Solution 1
          | Day 0 | Day 1 | Day 2
---------------------------------
Morning   |   2   |   0   |   0  
Afternoon |   3   |   1   |   2  
Evening   |   0   |   3   |   1  

Solution 2
          | Day 0 | Day 1 | Day 2
---------------------------------
Morning   |   2   |   0   |   0  
Afternoon |   3   |   1   |   2  
Evening   |   1   |   3   |   1  

Solution 3
          | Day 0 | Day 1 | Day 2
---------------------------------
Morning   |   1   |   1   |   2  
Afternoon |   3   |   3   |   0  
Evening   |   2   |   0   |   1  

Solution 4
          | Day 0 | Day 1 | Day 2
---------------------------------
Morning   |   1   |   0   |   2  
Afternoon |   3   |   3   |   0  
Evening   |   2   |   1   |   1  

Solution 5
          | Day 0 | Day 1 | Day 2
---------------------------------
Morning   |   2   |   0   |   2  
Afternoon |   3   |   2   |   0  
Evening   |   1   |   3   |   1  

Stop search after 5 solutions


<CpSolverStatus.FEASIBLE: 2>

### **2.5 Upgrading the Scheduler**


Now, you're going to be modifying and adding a few more constraints to this scheduling problem to account for even more complexity that can arise with real world scheduling problems. You'll be working with a small amount of data (only a handful of nurses over a few days) - but the beauty of constraint programming is that this *scales*. Sure you could spot check the schedules for a few nurses over a few days - but what about for a large hospital? They have hundreds of nurses, need to plan shifts weeks in advance, and need to account for a whole host of criteria such as experience, skill, and demand. This is where automated scheduling comes in handy!

The new constraints you'll be adding are as follows...



- **Modified Constraint 1: Variable Demand per Shift**
    - A majority of the time, a shift needs more than one nurse. `demand` is a 2D list, where `demand[day][shift]` gives you how many nurses need to be present for a shift on that certain day. Your solver should account for this, and **assign the exact number of nurses needed per shift to meet demand.**
    - **HINT:** This involves slightly changing a constraint that already exists, you shouldn't need to make a new one from scratch.
- **New Constraint 2: No Lone Junior Nurses**
    - Nurses have a seniority level, **a junior nurse must always have a senior nurse on shift to help train them**. Senior nurses can take shifts by themselves. `seniority[nurse]` gives the seniority of a given `nurse`: `seniority[nurse] = 0` if they are a junior nurse and `seniority[nurse] = 1` if they are a senior nurse. In the solution printer, juniors will be marked with an `*`.
    - **HINT:** I would recommend doing this by finding out how to put a constraint on all the senior nurses in a shift.
- **New Constraint 3: No Clopens**
    - If you've worked in the service industry, you may be (unfortunately) familiar with the concept of "clopen" shifts. This is when you have the closing/evening shift on one day, and then have to work the opening/morning shift the next day. If you're assigned a clopen, you have little time for rest in between - and repeatedly working clopens can have serious physical and mental health impacts due to lack of rest. Some jurisdictions have laws regulating clopen shifts.
    - For a small problem such as this, it's very easy to spot-check and see when a clopen occurs. However if you imagine expanding this to a large hospital with hundreds of nurses, scheduling weeks in advance - it is very likely that if this isn't accounted for a clopen may fall through the cracks.
    - **Your scheduler should never present a solution where a nurse works the evening shift on day `d`, and then has to work the morning shift on day `d+1`.**
    - **HINT:** You should do this by putting constraints on sets containing only two variables. What do those sets look like?

Finally, you should keep the constraints that the nurses work only one shift a day, and that their load is spread out evenly over the days (e.g. no nurse works fewer than the minimum number of shifts, and no more than the maximum number of shifts)


Just like the above examples, you'll be able to implement all of these using constraints of the form:
- [`model.add( sum(vars) == n )`](https://or-tools.github.io/docs/pdoc/ortools/sat/python/cp_model.html#CpModel.add)
- [`model.add_at_most_one(vars)`](https://or-tools.github.io/docs/pdoc/ortools/sat/python/cp_model.html#CpModel.add_at_most_one)
- [`model.add_exactly_one(vars)`](https://or-tools.github.io/docs/pdoc/ortools/sat/python/cp_model.html#CpModel.add_exactly_one)
- [`model.add_at_least_one(vars)`](https://or-tools.github.io/docs/pdoc/ortools/sat/python/cp_model.html#CpModel.add_at_least_one)

Creating the new constraints shouldn't take more than a dozen or so lines for each of the above constraints. The challenging part is figuring out how to subset the variables (and navigating nested data). I would recommend trying to draw out and visualize the subsets of data before writing code.

Here are the setup variables:

In [710]:
num_nurses = 6
num_shifts = 3
num_days = 3
all_nurses = range(num_nurses)
all_shifts = range(num_shifts)
all_days = range(num_days)

# How many nurses are needed for [day][shift]
demand = [
    [2,1,2],
    [2,1,3],
    [1,1,2],
]

# The seniority of nurse[n]
# 0 is junior, 1 is senior
seniority = [0,0,1,1,1,1]

In [711]:
# Setting up how many shifts need filling
total_shifts_need_filled = sum([sum(x) for x in demand])
print("Shifts needing to be filled: ", total_shifts_need_filled)

min_shifts_per_nurse = total_shifts_need_filled // num_nurses

if total_shifts_need_filled % num_nurses == 0:
    max_shifts_per_nurse = min_shifts_per_nurse
else:
    max_shifts_per_nurse = min_shifts_per_nurse + 1

print("Min shifts per nurse: ", min_shifts_per_nurse)
print("Max shifts per nurse: ",max_shifts_per_nurse)

Shifts needing to be filled:  15
Min shifts per nurse:  2
Max shifts per nurse:  3


We've created a fresh model for you below, and you should add constraints one at a time in the following cells. You are encouraged (and should) copy some of the existing code that was provided above as a starting point. We recommend frequently viewing printouts, as well as printing out the sets you're trying to put constraints on (like you did in the written exercises). We provided you with a solution printer (that also takes into account seniority).

The autograder works by running the solver to generate the first 100 solutions and to check if they satisfy the constraints. If those are all clean, you pass the autograder. If your constraints aren't set up correctly, it will probably give a mistaken solution within the first dozen or so solutions the solver prints.

**Remember to Restart + Run All if you delete your current constraint code and start again. You might *think* you deleted your constraints, but they might still be in memory!**

In [712]:
model = cp_model.CpModel()

shifts = {}

for n in all_nurses:
    for d in all_days:
        for s in all_shifts:
            shifts[(n, d, s)] = model.new_bool_var(f"shift_n{n}_d{d}_s{s}")

print(shifts)
# Below you will add the upgraded scheduling constraints one at a time

{(0, 0, 0): shift_n0_d0_s0(0..1), (0, 0, 1): shift_n0_d0_s1(0..1), (0, 0, 2): shift_n0_d0_s2(0..1), (0, 1, 0): shift_n0_d1_s0(0..1), (0, 1, 1): shift_n0_d1_s1(0..1), (0, 1, 2): shift_n0_d1_s2(0..1), (0, 2, 0): shift_n0_d2_s0(0..1), (0, 2, 1): shift_n0_d2_s1(0..1), (0, 2, 2): shift_n0_d2_s2(0..1), (1, 0, 0): shift_n1_d0_s0(0..1), (1, 0, 1): shift_n1_d0_s1(0..1), (1, 0, 2): shift_n1_d0_s2(0..1), (1, 1, 0): shift_n1_d1_s0(0..1), (1, 1, 1): shift_n1_d1_s1(0..1), (1, 1, 2): shift_n1_d1_s2(0..1), (1, 2, 0): shift_n1_d2_s0(0..1), (1, 2, 1): shift_n1_d2_s1(0..1), (1, 2, 2): shift_n1_d2_s2(0..1), (2, 0, 0): shift_n2_d0_s0(0..1), (2, 0, 1): shift_n2_d0_s1(0..1), (2, 0, 2): shift_n2_d0_s2(0..1), (2, 1, 0): shift_n2_d1_s0(0..1), (2, 1, 1): shift_n2_d1_s1(0..1), (2, 1, 2): shift_n2_d1_s2(0..1), (2, 2, 0): shift_n2_d2_s0(0..1), (2, 2, 1): shift_n2_d2_s1(0..1), (2, 2, 2): shift_n2_d2_s2(0..1), (3, 0, 0): shift_n3_d0_s0(0..1), (3, 0, 1): shift_n3_d0_s1(0..1), (3, 0, 2): shift_n3_d0_s2(0..1), (3, 1, 0)

#### **Exercise 2.5.1: Variable Demand per Shift**

The `demand` table says exactly how many nurses must be on each shift. Replace the
old "one nurse per shift" rule with a constraint that staffs each shift with exactly
`demand[day][shift]` nurses. Make sure you keep the "at most one shift per nurse per day" rule and the
balanced workload (min/max shifts per nurse) rules, and feel free to copy them over from above.

In [713]:
# Nurse can work only one shift a day (kept)
for n in all_nurses:
    for d in all_days:

        shifts_worked = []

        for s in all_shifts:
            shifts_worked.append(shifts[(n, d, s)])
        
        model.add_at_most_one(shifts_worked)

# A shift is staffed by exactly demand[day][shift] nurses (modified constraint 1)
# A shift is assigned to exactly one nurse
for d in all_days:
    for s in all_shifts:

        shifts_worked = []
        for n in all_nurses:
            shifts_worked.append(shifts[(n, d, s)])

        count = demand[d][s]
        model.add(sum(shifts_worked) == count)

# Each nurse works a balanced number of shifts (kept)
for n in all_nurses:
    shifts_worked = []
    
    for d in all_days:
        for s in all_shifts:
            shifts_worked.append(shifts[(n, d, s)])

    # Add the constraints that min_shifts_per_nurse <= sum(shifts_worked) <= max_shifts_per_nurse
    # Nurse can work no less than min shifts, and no more than max shifts.
    model.add(min_shifts_per_nurse <= sum(shifts_worked))
    model.add(sum(shifts_worked) <= max_shifts_per_nurse)


In [714]:
# Use this code to check your solutions - feel free to copy and paste it.
solver = cp_model.CpSolver()
solver.parameters.enumerate_all_solutions = True
solution_limit = 5

solution_printer = NursesCalendarPrinter(
    shifts, num_nurses, num_days, num_shifts, solution_limit,
    seniority=seniority,
)
solver.solve(model, solution_printer)

Solution 1
          |  Day 0  |  Day 1  |  Day 2 
---------------------------------------
Morning   |   3,5   |   2,3   |    3   
Afternoon |    2    |    5    |    2   
Evening   |  0*,1*  | 0*,1*,4 |   4,5  
(* = junior)

Solution 2
          | Day 0  | Day 1  | Day 2 
------------------------------------
Morning   |  2,4   |  0*,2  |   2   
Afternoon |   3    |   5    |   3   
Evening   |  1*,5  | 1*,3,4 |  0*,4 
(* = junior)

Solution 3
          | Day 0 | Day 1 | Day 2
---------------------------------
Morning   |  1*,2 |  0*,5 |   2  
Afternoon |   3   |   1*  |   3  
Evening   |  4,5  | 2,3,4 |  0*,4
(* = junior)

Solution 4
          | Day 0 | Day 1 | Day 2
---------------------------------
Morning   |  1*,2 |  0*,4 |   2  
Afternoon |   3   |   1*  |   3  
Evening   |  4,5  | 2,3,5 |  0*,4
(* = junior)

Solution 5
          | Day 0  | Day 1  | Day 2 
------------------------------------
Morning   |  2,4   |  4,5   |   2   
Afternoon |   3    |   1*   |   3   
Evening   |  1*,

<CpSolverStatus.FEASIBLE: 2>

In [715]:
grader.check("q_variable_demand")

q_variable_demand results: All test cases passed!

#### **Exercise 2.5.2: No Lone Junior Nurses**

You may notice that some of your above solutions have solo junior nurses. A junior nurse (`seniority[n] == 0`) must always share their shift with at least one
senior nurse (`seniority[n] == 1`). Add constraints so every shift satisfies this.

In [716]:
# All shifts must have a senior nurse on them (new constraint 2)
for d in all_days:
    for s in all_shifts:

        shifts_worked = []
        senior_nurses = []
        for i, n in enumerate(all_nurses):
            shifts_worked.append(shifts[(n, d, s)])
            if seniority[i] == 1:
                senior_nurses.append(shifts[(n, d, s)])
        model.add(sum(senior_nurses) >= 1)




In [717]:
# Use this code to check your solutions - feel free to copy and paste it.
solver = cp_model.CpSolver()
solver.parameters.enumerate_all_solutions = True
solution_limit = 5

solution_printer = NursesCalendarPrinter(
    shifts, num_nurses, num_days, num_shifts, solution_limit,
    seniority=seniority,
)
solver.solve(model, solution_printer)

Solution 1
          | Day 0  | Day 1  | Day 2 
------------------------------------
Morning   |  0*,5  |  0*,2  |   2   
Afternoon |   3    |   5    |   5   
Evening   |  1*,4  | 1*,3,4 |  0*,4 
(* = junior)

Solution 2
          | Day 0  | Day 1  | Day 2 
------------------------------------
Morning   |  4,5   |  1*,3  |   3   
Afternoon |   3    |   2    |   4   
Evening   |  1*,2  | 0*,4,5 |  0*,2 
(* = junior)

Solution 3
          |  Day 0  |  Day 1  |  Day 2 
---------------------------------------
Morning   |   4,5   |   4,5   |    5   
Afternoon |    3    |    3    |    2   
Evening   |   1*,2  | 0*,1*,2 |   0*,4 
(* = junior)

Solution 4
          |  Day 0  |  Day 1  |  Day 2 
---------------------------------------
Morning   |   4,5   |   4,5   |    5   
Afternoon |    3    |    3    |    2   
Evening   |   1*,2  | 0*,1*,2 |   0*,3 
(* = junior)

Solution 5
          | Day 0  | Day 1  | Day 2 
------------------------------------
Morning   |  1*,5  |  1*,4  |   4   
Afternoo

<CpSolverStatus.FEASIBLE: 2>

In [718]:
grader.check("q_no_lone_juniors")

q_no_lone_juniors results: All test cases passed!

#### **Exercise 2.5.3: No Clopens**

Almost there! However you now notice that in your solutions some unlucky nurses have to clopen. A **clopen** happens when a nurse works the evening (last) shift on day `d` and then the
morning (first) shift on day `d + 1`. Add constraints so that this never happens.

In [719]:
# No clopens: no evening shift on day d followed by morning shift on day d+1 (new constraint 3)
MORNING = 0
EVENING = 2

for n in all_nurses:
    for d in range(num_days - 1):
        model.Add(shifts[(n, d, EVENING)] + shifts[(n, d + 1, MORNING)] <= 1)



In [720]:
# Use this code to check your solutions - feel free to copy and paste it.
solver = cp_model.CpSolver()
solver.parameters.enumerate_all_solutions = True
solution_limit = 5

solution_printer = NursesCalendarPrinter(
    shifts, num_nurses, num_days, num_shifts, solution_limit,

)
solver.solve(model, solution_printer)

Solution 1
          | Day 0 | Day 1 | Day 2
---------------------------------
Morning   |  0,5  |  0,2  |   2  
Afternoon |   3   |   5   |   5  
Evening   |  1,4  | 1,3,4 |  0,4 

Solution 2
          | Day 0 | Day 1 | Day 2
---------------------------------
Morning   |  1,5  |  1,2  |   2  
Afternoon |   2   |   3   |   5  
Evening   |  0,3  | 0,4,5 |  0,4 

Solution 3
          | Day 0 | Day 1 | Day 2
---------------------------------
Morning   |  1,5  |  1,2  |   2  
Afternoon |   3   |   5   |   5  
Evening   |  0,4  | 0,3,4 |  0,4 

Solution 4
          | Day 0 | Day 1 | Day 2
---------------------------------
Morning   |  1,3  |  1,2  |   2  
Afternoon |   5   |   5   |   5  
Evening   |  0,4  | 0,3,4 |  0,4 

Solution 5
          | Day 0 | Day 1 | Day 2
---------------------------------
Morning   |  1,3  |  1,2  |   2  
Afternoon |   5   |   5   |   5  
Evening   |  0,4  | 0,3,4 |  1,4 

Stop search after 5 solutions


<CpSolverStatus.FEASIBLE: 2>

In [721]:
grader.check("q_no_clopens")

q_no_clopens results: All test cases passed!

## **3. (Written): HotSOS Struggles (10 pts)**

Hospitality management (i.e. managing resorts and hotels) is a complex industry that relies on a lot of moving parts. Guests are constantly moving in and out, rooms need to be booked and canceled, and when guests leave - rooms need to be cleaned. One industry standard hotel operations platform is Amadeus' [HotSOS](https://www.amadeus-hospitality.com/service-optimization-software/hotsos/), which provides a unified view of operations for managers and staff and can automate scheduling of what rooms need to be cleaned and when. One thing I would like to point out is that this software uses very little (if any) machine learning and no fancy generative AI - these are standard scheduling algorithms that are augmented by real-time data wrapped up in a holistic interface.

First, read this [article](https://www.amadeus-hospitality.com/insight/beyond-the-clipboard-the-future-of-hotel-housekeeping-software/) from the makers of HotSOS that explains the benefits and use cases of their software. Then, read the article ["More stress, fewer breaks: Hotel housekeepers reveal what it’s like working for an app"](https://www.marketplace.org/story/2026/05/14/hotel-housekeepers-say-aidriven-app-makes-job-tougher). This article tells the story about how the introduction of HotSOS has affected the daily routine of housekeepers.

Do the following:

- In the HotSOS article, summarize what performance metrics are being measured.
- After reading "More Stress, Fewer Breaks" - summarize how the introduction of the app impacted housekeeping staff and their day to day routines.
- Spend a few sentences commenting on both of the above. How did the performance measures presented in the HotSOS article interact with the day-to-day routine of housekeepers? Would you consider these unintended consequences? How should software integrate input from those in management positions and those with on-the-ground experience?


<!-- BEGIN QUESTION -->

- The Amadeus article emphasizes response times, labor efficiency, workload distribution, and service trends. Managers can monitor assignments, estimated labor time, coverage areas, and performance targets to adjust staffing and identify delays. Its main numerical example is a reported 31% improvement in response times at Hotel Okura Amsterdam. The article also presents guest satisfaction and staff well-being as intended benefits, but does not provide specific measurements for those outcomes. Amadeus article
  
- More Stress, Fewer Breaks describes how the app reduced some workers’ control over the order in which they cleaned rooms. Changing assignments sent them across floors with heavy carts, adding travel and physical strain. One housekeeper described a daily quota of 420 credits, roughly corresponding to expected cleaning minutes, that did not adequately account for unusually dirty rooms. She reported logging breaks while continuing to work because she feared missing her quota, and having less time for detailed cleaning. Experiences were not uniformly negative: another worker appreciated the assignment list and easier communication with her supervisor. Union negotiations later restored some workers’ ability to sequence their rooms and remain near their stations. Proof News article

- Together, these accounts suggest that measures intended to improve efficiency can encourage rushing and disrupt practical routines when they overlook travel time, room conditions, and rest. I would consider these plausible unintended consequences of a narrow definition of productivity, although the articles do not establish what designers or managers intended. Software development should involve managers in defining staffing and service goals and housekeepers in setting realistic time estimates, routing rules, and exceptions. Joint testing should evaluate cleaning quality, actual breaks, and physical workload alongside speed, with workers able to challenge assignments and influence revisions.

<!-- END QUESTION -->

## **Collaboration Statement** **(10 points)**

You are required to complete the following collaboration statement.


<!-- BEGIN QUESTION -->

- If you completed the lab in a group - list all your group members here, including yourself. **If you worked individually, you must state *"I worked by myself."***
    - I worked by myself.
- You must list the names of any others you got help from when completing this assignment (including TAs), and in what way you worked with them (discussed ideas, debugged math). **If you did not collaborate at all, you must state *"I did not collaborate with anyone else."***
    - I did not collaborate with anyone else.
- If you found any resources online helpful, you must link them and describe how you found them helpful (or not helpful). **If you did not use any resources online, you must state *"I did not use any resources online."***
    - I used google to search up methods for the model. It was helpful because it showed me what parameters I can put in.
- If you used any AI assistive technologies, you must describe how you used them and how you found them helpful (or not helpful). **If you did not use any AI assistive technologies, you must state *"I did not use any AI assistive technologies."***
    - I used some AI technology while searching up the methods for the model.
- If you have any questions or comments about the material, or there was a topic or question you are confused on -- please make a note of it here! **If you do not have any questions or comments, you must state *"I do not have any questions or comments about the assignment."***
    - I do not have any questions or comments about the assignment.
- Estimate how long this assignment took you in hours. **You must give an estimate**.
    - This assignment took me about 5 hours to complete.


<!-- END QUESTION -->



## Submission

Make sure you have run all cells in your notebook in order before running the cell below, so that all images/graphs appear in the output. The cell below will generate a zip file for you to submit. **Please save before exporting!**

Run every cell from top to bottom and SAVE before exporting. Then submit
the generated .zip to the autograder assignment on Gradescope.


In [722]:
# Save your notebook first, then run this cell to export your submission.
grader.export(pdf=False, run_tests=True)

Running your submission against local test cases...


Your submission received the following results when run against available test cases:

    q_variable_demand results: All test cases passed!

    q_no_lone_juniors results: All test cases passed!

    q_no_clopens results: All test cases passed!
